# NB_lab_test_trends

Builds `hcp_databricks_workspace.gold.lab_test_trends` per the STTM (Silver to Gold, v1.0): one row per test type per calendar month, driven by `silver.lab_results`, with an optional calendar join for month/quarter names and a LAG() month-over-month abnormal-rate trend. Load type: FULL rebuild.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hcp_databricks_workspace.gold

In [0]:
%sql
CREATE OR REPLACE TABLE hcp_databricks_workspace.gold.lab_test_trends
COMMENT 'Gold: one row per lab test type per month with abnormal/critical rates and month-over-month trend (STTM silver to gold v1.0)'
AS
WITH monthly AS (
  -- Driving table: lab_results grouped by test_name, year, month
  SELECT
    test_name,
    year(test_date)                            AS year,
    month(test_date)                           AS month,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count
  FROM hcp_databricks_workspace.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
with_rates AS (
  SELECT
    m.*,
    CAST(round(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(round(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_trend AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM with_rates r
)
SELECT
  t.test_name                                                     AS test_name,
  CAST(t.year AS INT)                                             AS year,
  CAST(t.month AS INT)                                            AS month,
  c.month_name                                                    AS month_name,
  c.quarter_name                                                  AS quarter_name,
  CAST(t.total_tests AS INT)                                      AS total_tests,
  CAST(t.abnormal_test_count AS INT)                              AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                              AS critical_test_count,
  t.abnormal_rate_pct                                             AS abnormal_rate_pct,
  t.critical_rate_pct                                             AS critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))            AS prev_month_abnormal_rate_pct,
  CASE
    WHEN t.prev_month_abnormal_rate_pct IS NULL THEN NULL
    ELSE CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                                             AS abnormal_rate_change_pct   -- percentage-point change
FROM with_trend t
LEFT JOIN hcp_databricks_workspace.silver.calendar c
  ON c.datekey = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT)   -- first day of month; calendar key column is `datekey`

In [0]:
%sql
-- Grain and reconciliation checks against silver
SELECT
  count(*)                                                        AS gold_rows,
  count(DISTINCT test_name, year, month)                          AS distinct_test_months,
  count(DISTINCT test_name)                                       AS test_types,
  sum(total_tests)                                                AS gold_total_tests,
  (SELECT count(*) FROM hcp_databricks_workspace.silver.lab_results) AS silver_total_tests,
  sum(abnormal_test_count)                                        AS abnormal_tests,
  sum(critical_test_count)                                        AS critical_tests,
  count_if(month_name IS NULL)                                    AS missing_calendar_match,
  count_if(prev_month_abnormal_rate_pct IS NULL)                  AS first_month_rows
FROM hcp_databricks_workspace.gold.lab_test_trends

In [0]:
%sql
SELECT * FROM hcp_databricks_workspace.gold.lab_test_trends ORDER BY test_name, year, month LIMIT 24